In [8]:
# Cell 1 — Environment setup & imports

import os
from pathlib import Path

from ultralytics import YOLO

PROJECT_ROOT = Path("../../smart-object-detection").resolve()  # notebook lives in notebooks/, project is one level up
DATA_CONFIG = PROJECT_ROOT / "configs" / "data" / "mug.yaml"

print("Project root:", PROJECT_ROOT)
print("Data config:", DATA_CONFIG)
print("Exists?", DATA_CONFIG.exists())


Project root: D:\an4sem1\PRS\smart-object-detection
Data config: D:\an4sem1\PRS\smart-object-detection\configs\data\mug.yaml
Exists? True


In [9]:
# Cell 2 — Sanity check: show mug.yaml content

import yaml

with open(DATA_CONFIG, "r") as f:
    cfg = yaml.safe_load(f)

print(cfg)


{'path': 'D:/an4sem1/PRS/smart-object-detection/mug_coco_yolo', 'train': 'images/train2017', 'val': 'images/val2017', 'names': {0: 'mug'}}


In [10]:
# Cell 3 — Define training parameters

MODEL_NAME = "yolov8n.pt"  # you can switch to yolov8s.pt later
EPOCHS = 25
IMG_SIZE = 512
BATCH_SIZE = 8
EXPERIMENT_NAME = "mug_yolov8n_notebook"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUTPUT_DIR)


Output dir: D:\an4sem1\PRS\smart-object-detection\outputs


In [ ]:
# Cell 4 — Create and train YOLOv8 model

model = YOLO(MODEL_NAME)  # load COCO-pretrained YOLOv8n

results = model.train(
    data=str(DATA_CONFIG),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(OUTPUT_DIR),
    workers=2,
    cache="disk",
    name=EXPERIMENT_NAME,
    exist_ok=True,
)

results


New https://pypi.org/project/ultralytics/8.3.249 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.7 torch-2.9.1+cpu CPU (12th Gen Intel Core(TM) i5-12450H)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=D:\an4sem1\PRS\smart-object-detection\configs\data\mug.yaml, epochs=25, time=None, patience=100, batch=8, imgsz=512, save=True, save_period=-1, cache=disk, device=None, workers=2, project=D:\an4sem1\PRS\smart-object-detection\outputs, name=mug_yolov8n_notebook, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=Fals

train: Scanning D:\an4sem1\PRS\smart-object-detection\mug_coco_yolo\labels\train2017... 13303 images, 6500 backgrounds, 0 corrupt: 100%|██████████| 13303/13303 [00:50<00:00, 264.18it/s]


train: New cache created: D:\an4sem1\PRS\smart-object-detection\mug_coco_yolo\labels\train2017.cache


train: Caching images (10.4GB Disk): 100%|██████████| 13303/13303 [01:04<00:00, 205.23it/s]
D:\an4sem1\PRS\smart-object-detection\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning D:\an4sem1\PRS\smart-object-detection\mug_coco_yolo\labels\val2017... 1390 images, 1000 backgrounds, 0 corrupt: 100%|██████████| 1390/1390 [00:05<00:00, 249.45it/s]


val: New cache created: D:\an4sem1\PRS\smart-object-detection\mug_coco_yolo\labels\val2017.cache


val: Caching images (1.1GB Disk): 100%|██████████| 1390/1390 [00:06<00:00, 221.30it/s]


Plotting labels to D:\an4sem1\PRS\smart-object-detection\outputs\mug_yolov8n_notebook\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
Image sizes 512 train, 512 val
Using 0 dataloader workers
Logging results to D:\an4sem1\PRS\smart-object-detection\outputs\mug_yolov8n_notebook
Starting training for 25 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/25         0G      1.274      4.044      1.124         13        512:   8%|▊         | 138/1663 [04:38<49:11,  1.94s/it]

In [ ]:
# Cell 5 — Evaluate model on validation set (YOLO built-in)

best_pt = OUTPUT_DIR / EXPERIMENT_NAME / "weights" / "best.pt"
model = YOLO(str(best_pt))
val_results = model.val(data=str(DATA_CONFIG), imgsz=IMG_SIZE)
val_results


In [ ]:
# Cell 6 — Test prediction on a few validation images and visualize

VAL_IMAGES_DIR = PROJECT_ROOT / "mug_coco_yolo" / "images" / "val2017"
print("Val images dir:", VAL_IMAGES_DIR, " | Exists:", VAL_IMAGES_DIR.exists())

sample_source = str(VAL_IMAGES_DIR)  # folder of images

pred_results = model.predict(
    source=sample_source,
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,  # saves images with boxes
    project=str(OUTPUT_DIR),
    name=f"{EXPERIMENT_NAME}_val_pred",
    exist_ok=True,
)

pred_results[:3]


In [ ]:
# Cell 7 — See where the results were saved

print("YOLO training runs:")
for p in (OUTPUT_DIR).glob("*"):
    print("  ", p)

print("\nCheck also:")
print("  -", OUTPUT_DIR / EXPERIMENT_NAME)
print("  -", OUTPUT_DIR / f"{EXPERIMENT_NAME}_val_pred")
